# SACDpy batch reconstruction pipeline

This notebook runs SACD reconstruction from TIFF stacks using one parameter setup. It works for a single input TIFF or a folder of TIFF files. If `frames_per_sacd` is set, each input stack can produce a SACD time series.

## 1. Imports and paths

Run this notebook from the SACDpy package folder. If the package has not been installed with `pip install -e .`, the local `src/` folder is added to the Python path.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
src = repo / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import matplotlib.pyplot as plt
import numpy as np

from sacdpy import SACDParams, read_tiff_stack, reconstruct, write_tiff_image

print(f"Working folder: {repo}")

## 2. Input and output setup

Set `input_path` to either one TIFF file or a folder. When it is a folder, `glob_pattern` controls which TIFFs are processed. Names containing any text in `exclude_name_contains` are skipped.

In [ ]:
input_path = repo / "tests" / "testdata"  # folder or single .tif/.tiff file
glob_pattern = "*.tif"
exclude_name_contains = ("SACD",)  # useful when a folder also contains processed outputs

output_dir = repo / "outputs"
output_suffix = "_SACDpy"
output_dir.mkdir(parents=True, exist_ok=True)

def find_input_files(path):
    path = Path(path)
    if path.is_file():
        return [path]
    files = sorted(path.glob(glob_pattern))
    return [
        f for f in files
        if f.is_file() and not any(token in f.name for token in exclude_name_contains)
    ]

input_files = find_input_files(input_path)
if not input_files:
    raise FileNotFoundError(f"No input TIFF files found from {input_path}")

for f in input_files:
    print(f.name)

## 3. SACD parameter setup

Set microscope and SACD parameters here. `frames_per_sacd=None` uses the full stack to create one SACD image. For example, `frames_per_sacd=25` turns a 50-frame input into two SACD output frames. Leftover partial chunks are dropped.

In [ ]:
pixel_nm = 117.0
na = 1.45
default_wavelength_nm = 561

# Optional filename-based wavelength overrides for mixed-channel folders.
wavelength_by_name = {
    "left": 561,
    "right": 640,
}

mag = 2
iter1 = 7
iter2 = 8
ac_order = 2
subfactor = 0.8
frames_per_sacd = None  # set to 25, for example, to create SACD frames from 25-frame chunks

def wavelength_for_file(path):
    name = path.name.lower()
    for token, wavelength in wavelength_by_name.items():
        if token.lower() in name:
            return wavelength
    return default_wavelength_nm

def params_for_file(path):
    return SACDParams(
        pixel_nm=pixel_nm,
        wavelength_nm=wavelength_for_file(path),
        na=na,
        mag=mag,
        iter1=iter1,
        iter2=iter2,
        ac_order=ac_order,
        subfactor=subfactor,
        frames_per_sacd=frames_per_sacd,
    )

## 4. Batch processing

Each input TIFF is loaded, reconstructed, and saved as a float32 TIFF. A chunked reconstruction with more than one SACD frame is saved as a time-first TIFF stack.

In [ ]:
def output_path_for_file(path):
    return output_dir / f"{path.stem}{output_suffix}.tif"

results = []
for input_file in input_files:
    params = params_for_file(input_file)
    stack = read_tiff_stack(input_file)
    sacd = reconstruct(stack, params)
    output_file = output_path_for_file(input_file)
    write_tiff_image(output_file, sacd)
    results.append({
        "input": input_file,
        "output": output_file,
        "input_shape": stack.shape,
        "output_shape": sacd.shape,
        "wavelength_nm": params.wavelength_nm,
        "frames_per_sacd": params.frames_per_sacd,
    })
    print(f"{input_file.name} -> {output_file.name} | {stack.shape} -> {sacd.shape}")

## 5. Quick output preview

For SACD videos, the first output frame is shown. Open the saved TIFF in your preferred TIFF viewer to inspect the full stack.

In [ ]:
def preview_image(img):
    if img.ndim == 2:
        return img
    return img[0]

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
for ax, item in zip(axes.ravel(), results):
    img = read_tiff_stack(item["output"])
    view = preview_image(img)
    ax.imshow(view, cmap="hot")
    ax.set_title(f"{item['output'].name}\nshape={item['output_shape']}")
    ax.axis("off")
plt.tight_layout();

## 6. Processing summary

In [ ]:
for item in results:
    print(
        f"input={item['input'].name} | output={item['output']} | "
        f"input_shape={item['input_shape']} | output_shape={item['output_shape']} | "
        f"wavelength_nm={item['wavelength_nm']} | frames_per_sacd={item['frames_per_sacd']}"
    )